In [8]:
import pandas as pd
from surprise import SVD, Dataset, Reader

# Dataset dos Amigos com os TÓPICOS ANTIGOS + NOVOS TÓPICOS
dados_amigos = {
    'usuario': [
        # --- Tópicos Antigos (Esportes, IA, Literatura, Estudos) ---
        'Guilherme', 'Guilherme', 'Guilherme',
        'William', 'William', 'William',
        'Carol', 'Carol', 'Carol',
        'Joao', 'Joao',
        'David', 'David',
        'Isabela', 'Isabela', 'Isabela',
        'Matheus', 'Matheus',
        'Mariane', 'Mariane', 'Mariane',
        
        # --- NOVOS TÓPICOS (Finanças, Cinema, Gastronomia, Games) ---
        'Guilherme', 'Guilherme', 'Guilherme',
        'William', 'William', 'William',
        'Carol', 'Carol', 'Carol',
        'Joao', 'Joao',
        'David', 'David',
        'Isabela', 'Isabela', 'Isabela',
        'Matheus', 'Matheus',
        'Mariane', 'Mariane', 'Mariane'
    ],
    'item': [
        # Itens Antigos
        'Treino Funcional & Calistenia', 'Curso de Inteligência Artificial', 'Livro: Duna',
        'Curso de Produtividade e Foco', 'Livro: O Poder do Hábito', 'Curso de Inteligência Artificial',
        'Livro: Orgulho e Preconceito', 'Curso de Idiomas (Inglês)', 'Aula de Yoga e Meditação',
        'Treino Funcional & Calistenia', 'Curso de Inteligência Artificial',
        'Livro: Duna', 'Treino Funcional & Calistenia',
        'Livro: Orgulho e Preconceito', 'Curso de Produtividade e Foco', 'Livro: O Poder do Hábito',
        'Treino Funcional & Calistenia', 'Curso de Inteligência Artificial',
        'Curso de Idiomas (Inglês)', 'Livro: Orgulho e Preconceito', 'Aula de Yoga e Meditação',
        
        # Novos Itens
        'Curso: Investimentos do Zero', 'Filme: Interestelar', 'Jogo: Elden Ring',
        'Curso: Investimentos do Zero', 'Workshop: Hambúrguer Artesanal', 'Série: Breaking Bad',
        'Curso de Confeitaria Francesa', 'Filme: La La Land', 'Jogo: The Sims / Animal Crossing',
        'Jogo: Elden Ring', 'Filme: Interestelar',
        'Filme: Interestelar', 'Jogo: Elden Ring',
        'Filme: La La Land', 'Curso de Confeitaria Francesa', 'Livro: Pai Rico, Pai Pobre',
        'Jogo: Elden Ring', 'Curso: Investimentos do Zero',
        'Curso de Confeitaria Francesa', 'Filme: La La Land', 'Série: Breaking Bad'
    ],
    'nota': [
        # Notas Antigas
        5, 4, 2, 5, 5, 3, 5, 4, 4, 4, 5, 5, 3, 4, 4, 3, 5, 5, 5, 3, 5,
        # Novas Notas
        5, 5, 4, # Guilherme
        4, 4, 5, # William
        5, 5, 4, # Carol
        5, 4,    # Joao
        5, 5,    # David
        4, 4, 5, # Isabela
        4, 5,    # Matheus
        5, 4, 5  # Mariane
    ]
}

df = pd.DataFrame(dados_amigos)

# Configurando o leitor de notas (1 a 5)
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['usuario', 'item', 'nota']], reader)

# Treinando na matriz completa
trainset = data.build_full_trainset()

# Inicializando o SVD (Aumentamos os fatores para capturar os novos tópicos)
model = SVD(n_factors=10, random_state=42)
model.fit(trainset)

# Função de Recomendação
def recomendar_para_amigo(nome_amigo, top_n=3):
    if nome_amigo not in df['usuario'].unique():
        return f"Usuário '{nome_amigo}' não encontrado."
    
    itens_avaliados = df[df['usuario'] == nome_amigo]['item'].tolist()
    todos_itens = df['item'].unique()
    itens_nao_avaliados = [item for item in todos_itens if item not in itens_avaliados]
    
    if not itens_nao_avaliados:
        return f"{nome_amigo} já avaliou tudo!"
    
    predicoes = []
    for item in itens_nao_avaliados:
        pred = model.predict(nome_amigo, item)
        predicoes.append((item, round(pred.est, 2)))
    
    predicoes.sort(key=lambda x: x[1], reverse=True)
    
    print(f"--- Top {top_n} Recomendações para: {nome_amigo} ---")
    for i, (item, nota_prevista) in enumerate(predicoes[:top_n], 1):
        print(f"{i}. {item} (Nota estimada: {nota_prevista})")

recomendar_para_amigo('Carol')

recomendar_para_amigo('Joao')

recomendar_para_amigo('Guilherme')

recomendar_para_amigo('Isabela')

recomendar_para_amigo('Matheus')

recomendar_para_amigo('Mariane')

recomendar_para_amigo('David')

--- Top 3 Recomendações para: Carol ---
1. Série: Breaking Bad (Nota estimada: 4.53)
2. Curso: Investimentos do Zero (Nota estimada: 4.49)
3. Filme: Interestelar (Nota estimada: 4.49)
--- Top 3 Recomendações para: Joao ---
1. Série: Breaking Bad (Nota estimada: 4.52)
2. Curso de Confeitaria Francesa (Nota estimada: 4.5)
3. Curso: Investimentos do Zero (Nota estimada: 4.48)
--- Top 3 Recomendações para: Guilherme ---
1. Série: Breaking Bad (Nota estimada: 4.47)
2. Curso de Confeitaria Francesa (Nota estimada: 4.37)
3. Livro: Pai Rico, Pai Pobre (Nota estimada: 4.35)
--- Top 3 Recomendações para: Isabela ---
1. Série: Breaking Bad (Nota estimada: 4.34)
2. Filme: Interestelar (Nota estimada: 4.32)
3. Curso de Idiomas (Inglês) (Nota estimada: 4.26)
--- Top 3 Recomendações para: Matheus ---
1. Filme: Interestelar (Nota estimada: 4.59)
2. Série: Breaking Bad (Nota estimada: 4.55)
3. Curso de Confeitaria Francesa (Nota estimada: 4.55)
--- Top 3 Recomendações para: Mariane ---
1. Livro: Pai Ri

In [7]:
from surprise.model_selection import GridSearchCV

param_grid = {'n_factors': [5, 10, 20], 'lr_all': [0.005, 0.01], 'reg_all': [0.02, 0.1]}
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3)
gs.fit(data)

print("Melhor RMSE:", gs.best_score['rmse'])
print("Melhores parâmetros:", gs.best_params['rmse'])

Melhor RMSE: 0.803649067855987
Melhores parâmetros: {'n_factors': 10, 'lr_all': 0.005, 'reg_all': 0.1}
